# 4.2. Población y migración de MySQL a MongoDB

Este notebook implementa la lectura, transformación y carga de los datos
de la base `salud_usm`, desde MySQL hacia MongoDB, utilizando Python.

Los datos de origen se obtienen del respaldo `BD/salud_usm.sql`,
previamente importado en MySQL. La transformación utiliza el modelo
documental definido en la sección 4.1.

El procedimiento se organiza en las siguientes etapas:

1. Comprobación del entorno y conexión con las bases de datos.
2. Lectura de las tablas de MySQL.
3. Transformación de registros y relaciones en documentos.
4. Inserción en MongoDB.
5. Generación de comprobantes y fichas.
6. Verificación de conteos, contenido y requisitos de población.

Antes de ejecutar el notebook, seguir las instrucciones de instalación
y despliegue del README.

Las celdas de carga requieren colecciones de destino vacías. Para
consultar una base ya poblada, no deben repetirse las inserciones.

### Comprobación del entorno Python

Se muestra la ubicación del intérprete y las versiones de PyMySQL
y PyMongo. Esto permite comprobar que el kernel seleccionado dispone
de las bibliotecas necesarias.

In [1]:
import sys
import pymysql
import pymongo
from importlib.metadata import version

print("Python utilizado:", sys.executable)
print("PyMySQL:", version("pymysql"))
print("PyMongo:", version("pymongo"))
print("Bibliotecas disponibles correctamente.")

Python utilizado: C:\Python313\python.exe
PyMySQL: 1.2.3
PyMongo: 4.19.0
Bibliotecas disponibles correctamente.


### Conexión con MySQL

Se establece una conexión con la base `salud_usm` del contenedor MySQL,
disponible en `127.0.0.1:3307`.

La contraseña se solicita mediante `getpass` para evitar escribirla
directamente en el código. Se utiliza un cursor de diccionarios para
acceder a cada columna mediante su nombre.

Esta celda establece la conexión; todavía no migra datos.

In [2]:
import pymysql
from getpass import getpass

clave_mysql = getpass("Constraseña de MySQL: ")

conexion = pymysql.connect(
    host="127.0.0.1",
    port=3307,
    user="root",
    password=clave_mysql,
    database="salud_usm",
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor,
    connect_timeout=5,
    autocommit=True
)

print("Conexión con MySQL establecida correctamente.")

Conexión con MySQL establecida correctamente.


### Lectura de las tablas de origen

Se consulta `INFORMATION_SCHEMA.TABLES` para obtener los nombres de
las tablas de `salud_usm`. Se excluyen las vistas porque sus resultados
se derivan de las tablas originales.

Para cada tabla se ejecutan dos consultas:

- `SELECT COUNT(*)`: obtiene la cantidad de registros.
- `SELECT *`: recupera su contenido.

Los resultados se almacenan en `conteos_mysql` y `datos_mysql`.
Estas variables son copias en memoria y no se actualizan automáticamente
si posteriormente cambia la base de datos.

In [3]:
datos_mysql = {}
conteos_mysql = {}

# Delimitar los nombres de tabla con comillas invertidas.
# Si un nombre contiene una comilla invertida, duplicarla.
def identificador_sql(nombre):
    return "`" + nombre.replace("`", "``") + "`"

with conexion.cursor() as cursor:
    cursor.execute(
        """
        SELECT TABLE_NAME AS tabla
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = %s
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
        """,
        ("salud_usm",)
    )

    tablas = [fila["tabla"] for fila in cursor.fetchall()]

    for tabla in tablas:
        nombre_sql = identificador_sql(tabla)

        cursor.execute(f"SELECT COUNT(*) AS total FROM {nombre_sql}")
        conteos_mysql[tabla] = cursor.fetchone()["total"]

        cursor.execute(f"SELECT * FROM {nombre_sql}")
        datos_mysql[tabla] = cursor.fetchall()

        print(f"{tabla}: {conteos_mysql[tabla]} registros")

atencion: 60 registros
atencion_diagnostico: 72 registros
centro_medico: 5 registros
centro_medico_medico: 110 registros
cita: 205 registros
diagnostico: 14 registros
especialidad: 8 registros
estado: 5 registros
medico: 22 registros
medico_especialidad: 59 registros
paciente: 49 registros
prevision: 3 registros
receta: 80 registros


### Inspección de una muestra de los datos

Se muestran hasta tres registros por tabla para revisar sus campos,
valores y tipos de datos. Esta muestra no representa el total
de registros ni constituye por sí sola una validación completa.

In [4]:
from IPython.display import display

for tabla, registros in datos_mysql.items():
    print(f"\nTabla: {tabla}")
    display(registros[:3])


Tabla: atencion


[{'Nro_Atencion': 1,
  'Motivo_Atencion': 'Fiebre y congestión nasal',
  'Observaciones_Atencion': 'Rinofaringitis; presión arterial elevada en control.',
  'Nro_Cita': 1},
 {'Nro_Atencion': 2,
  'Motivo_Atencion': 'Dolor de garganta y fiebre',
  'Observaciones_Atencion': 'Amígdalas inflamadas con exudado.',
  'Nro_Cita': 2},
 {'Nro_Atencion': 3,
  'Motivo_Atencion': 'Irritación cutánea en antebrazos',
  'Observaciones_Atencion': 'Dermatitis de contacto leve.',
  'Nro_Cita': 3}]


Tabla: atencion_diagnostico


[{'Nro_Atencion': 16, 'Codigo_Diagnostico': 'A09'},
 {'Nro_Atencion': 18, 'Codigo_Diagnostico': 'A09'},
 {'Nro_Atencion': 32, 'Codigo_Diagnostico': 'A09'}]


Tabla: centro_medico


[{'Codigo_Centro': 1,
  'Nombre_Centro': 'SaludUSM San Joaquín',
  'Comuna_Centro': 'San Joaquín',
  'Region_Centro': 'RM'},
 {'Codigo_Centro': 2,
  'Nombre_Centro': 'SaludUSM Vitacura',
  'Comuna_Centro': 'Vitacura',
  'Region_Centro': 'RM'},
 {'Codigo_Centro': 3,
  'Nombre_Centro': 'SaludUSM Valparaíso',
  'Comuna_Centro': 'Valparaíso',
  'Region_Centro': 'VA'}]

... (16 outputs left out)


Tabla: prevision


[{'ID_Prevision': 1, 'Prevision_Paciente': 'Fonasa'},
 {'ID_Prevision': 2, 'Prevision_Paciente': 'Isapre'},
 {'ID_Prevision': 3, 'Prevision_Paciente': 'Particular'}]


Tabla: receta


[{'ID_Receta': 1,
  'Medicamento_Receta': 'Paracetamol 500 mg',
  'Dosis_Receta': '1 comprimido cada 8 horas',
  'Dias-Tratamiento_Receta': 5,
  'Nro_Atencion': 1},
 {'ID_Receta': 2,
  'Medicamento_Receta': 'Losartán 50 mg',
  'Dosis_Receta': '1 comprimido cada 24 horas',
  'Dias-Tratamiento_Receta': 30,
  'Nro_Atencion': 1},
 {'ID_Receta': 3,
  'Medicamento_Receta': 'Amoxicilina 500 mg',
  'Dosis_Receta': '1 cápsula cada 8 horas',
  'Dias-Tratamiento_Receta': 7,
  'Nro_Atencion': 2}]

## Preparación de MongoDB

Se despliegan tres servidores MongoDB mediante Docker Compose.
Los servidores forman el replica set rsSaludUSM.

La configuración inicial y la comprobación del estado se realizan
desde Python con PyMongo. Este despliegue se utilizará para la
migración de la sección 4.2 y las pruebas de replicación de la sección 4.4.

#### Comprobación de los tres nodos

Se verifica que Windows pueda resolver los nombres `mongo1`, `mongo2`
y `mongo3`, y que cada servidor responda al comando `ping`.

Esta comprobación verifica la conectividad individual. Todavía no
confirma que el conjunto de réplicas esté correctamente configurado.

In [5]:
import socket
from pymongo import MongoClient

# Comprobar que Windows reconoce los nombres y MongoDB responde.
nodos = [
    ("mongo1", 30001),
    ("mongo2", 30002),
    ("mongo3", 30003),
]

for nombre, puerto in nodos:
    direccion = socket.gethostbyname(nombre)

    with MongoClient(
        f"mongodb://{nombre}:{puerto}/?directConnection=true",
        serverSelectionTimeoutMS=10000
    ) as cliente_nodo:
        cliente_nodo.admin.command("ping")

    print(f"{nombre}:{puerto} | IP: {direccion} | Conexión correcta")

mongo1:30001 | IP: 127.0.0.1 | Conexión correcta
mongo2:30002 | IP: 127.0.0.1 | Conexión correcta
mongo3:30003 | IP: 127.0.0.1 | Conexión correcta


#### Inicialización del conjunto de réplicas

Se define el conjunto `rsSaludUSM` con sus tres miembros.

Antes de inicializarlo, se consulta su estado. Si ya está configurado,
se conserva la configuración existente; si aún no está inicializado,
se solicita su creación.

In [6]:
from pymongo.errors import OperationFailure

configuracion = {
    "_id": "rsSaludUSM",
    "members": [
        {"_id": 0, "host": "mongo1:30001"},
        {"_id": 1, "host": "mongo2:30002"},
        {"_id": 2, "host": "mongo3:30003"},
    ]
}

with MongoClient(
    "mongodb://mongo1:30001/?directConnection=true",
    serverSelectionTimeoutMS=10000
) as cliente_inicial:

    try:
        # Consultar primero evita reinicializar un conjunto existente.
        cliente_inicial.admin.command("replSetGetStatus")
        print("El replica set ya estaba inicializado.")

    except OperationFailure as error:
        if error.code == 94:  # NotYetInitialized
            resultado = cliente_inicial.admin.command(
                "replSetInitiate",
                configuracion
            )
            print("Inicialización solicitada:", resultado["ok"])
        else:
            raise

Inicialización solicitada: 1.0


#### Conexión al conjunto de réplicas

Se crea un cliente que conoce las direcciones de los tres nodos
y puede descubrir cuál actúa como primario.

Las escrituras utilizan `write concern majority`, que solicita
confirmación de la mayoría correspondiente del conjunto de réplicas.

Finalmente, se muestran los roles de los miembros. Antes de continuar,
debe existir un `PRIMARY` y dos `SECONDARY`.

In [7]:
from pymongo.write_concern import WriteConcern

# Incluir los tres servidores permite descubrir cuál es el primario.
uri_mongo = (
    "mongodb://mongo1:30001,mongo2:30002,mongo3:30003/"
    "?replicaSet=rsSaludUSM"
)

cliente_mongo = MongoClient(
    uri_mongo,
    serverSelectionTimeoutMS=60000,
    tz_aware=True
)

# Esta consulta espera a que haya un primario disponible.
cliente_mongo.admin.command("ping")

# Preparar el acceso a la base con confirmación de escritura mayoritaria.
db_mongo = cliente_mongo.get_database(
    "salud_usm",
    write_concern=WriteConcern(w="majority", wtimeout=15000)
)

estado = cliente_mongo.admin.command("replSetGetStatus")

for miembro in estado["members"]:
    print(f"{miembro['name']} → {miembro['stateStr']}")

print("\nConexión a MongoDB establecida.")

mongo1:30001 → PRIMARY
mongo2:30002 → SECONDARY
mongo3:30003 → SECONDARY

Conexión a MongoDB establecida.


#### Resumen de los datos disponibles

Se muestran las cantidades de registros previamente leídos desde MySQL.
Este resumen utiliza los datos en memoria; no ejecuta nuevas consultas
sobre la base de origen.

In [8]:
# Confirmar los datos disponibles para la migración.
for tabla, registros in datos_mysql.items():
    print(f"{tabla}: {len(registros)} registros")

atencion: 60 registros
atencion_diagnostico: 72 registros
centro_medico: 5 registros
centro_medico_medico: 110 registros
cita: 205 registros
diagnostico: 14 registros
especialidad: 8 registros
estado: 5 registros
medico: 22 registros
medico_especialidad: 59 registros
paciente: 49 registros
prevision: 3 registros
receta: 80 registros


### Transformación de las relaciones del modelo relacional

Las tablas intermedias de MySQL se transformarán en arreglos dentro
de los documentos correspondientes:

- Cada médico incluirá los identificadores de sus centros y especialidades.
- Cada atención incluirá los códigos de sus diagnósticos.
- Cada atención incluirá sus recetas como un arreglo de documentos.

Se conservarán los identificadores originales para mantener las
asociaciones entre los registros.

En esta etapa solo se preparan estructuras de Python en memoria.
Todavía no se escriben documentos en MongoDB.

#### Agrupación de centros, especialidades, diagnósticos y recetas

In [9]:
from collections import defaultdict

# Agrupar las relaciones por el identificador de su documento principal.
centros_por_medico = defaultdict(list)
especialidades_por_medico = defaultdict(list)
diagnosticos_por_atencion = defaultdict(list)
recetas_por_atencion = defaultdict(list)

# Médicos y centros en los que trabajan.
for fila in datos_mysql["centro_medico_medico"]:
    centros_por_medico[fila["RUT_Medico"]].append(
        fila["Codigo_Centro"]
    )

# Médicos y especialidades que tienen asignadas.
for fila in datos_mysql["medico_especialidad"]:
    especialidades_por_medico[fila["RUT_Medico"]].append(
        fila["ID_Especialidad"]
    )

# Atenciones y sus diagnósticos.
for fila in datos_mysql["atencion_diagnostico"]:
    diagnosticos_por_atencion[fila["Nro_Atencion"]].append(
        fila["Codigo_Diagnostico"]
    )

# Cada receta conserva su identificador y sus datos originales.
for fila in datos_mysql["receta"]:
    receta = {
        "receta_id": fila["ID_Receta"],
        "medicamento": fila["Medicamento_Receta"],
        "dosis": fila["Dosis_Receta"],
        "dias": fila["Dias-Tratamiento_Receta"]
    }

    recetas_por_atencion[fila["Nro_Atencion"]].append(receta)

print("Relaciones y recetas agrupadas correctamente.")

Relaciones y recetas agrupadas correctamente.


#### Validación de las agrupaciones

Se compara la cantidad de relaciones y recetas agrupadas con los
conteos obtenidos desde MySQL. También se comprueba que cada atención
tenga al menos un diagnóstico.

Las instrucciones `assert` detienen la ejecución si alguna condición
no se cumple.

In [10]:
# Comparar lo leído desde MySQL con los elementos agrupados.
verificaciones = [
    (
        "Médico–centro",
        "centro_medico_medico",
        centros_por_medico
    ),
    (
        "Médico–especialidad",
        "medico_especialidad",
        especialidades_por_medico
    ),
    (
        "Atención–diagnóstico",
        "atencion_diagnostico",
        diagnosticos_por_atencion
    ),
    (
        "Recetas",
        "receta",
        recetas_por_atencion
    )
]

for nombre, tabla, grupos in verificaciones:
    total_origen = conteos_mysql[tabla]
    total_agrupado = sum(len(elementos) for elementos in grupos.values())

    assert total_origen == total_agrupado, (
        f"{nombre}: se esperaban {total_origen} "
        f"elementos y se obtuvieron {total_agrupado}"
    )

    print(
        f"{nombre}: MySQL = {total_origen}, "
        f"agrupados = {total_agrupado} → Correcto"
    )

# Comprobar que cada atención tiene al menos un diagnóstico.
atenciones_sin_diagnostico = [
    fila["Nro_Atencion"]
    for fila in datos_mysql["atencion"]
    if not diagnosticos_por_atencion.get(fila["Nro_Atencion"])
]

print("\nAtenciones sin diagnóstico:", len(atenciones_sin_diagnostico))

assert not atenciones_sin_diagnostico, (
    f"Faltan diagnósticos en las atenciones: {atenciones_sin_diagnostico}"
)

Médico–centro: MySQL = 110, agrupados = 110 → Correcto
Médico–especialidad: MySQL = 59, agrupados = 59 → Correcto
Atención–diagnóstico: MySQL = 72, agrupados = 72 → Correcto
Recetas: MySQL = 80, agrupados = 80 → Correcto

Atenciones sin diagnóstico: 0


### Construcción de los documentos

Se conservan los identificadores originales de MySQL en el campo _id.
Las referencias utilizan esos mismos valores y tipos de datos.

Las relaciones de médicos con centros y especialidades se representan
como arreglos de identificadores. Las atenciones contienen los códigos
de sus diagnósticos y un arreglo de recetas.

Las fechas sin hora se representan como fechas BSON a medianoche UTC,
utilizadas únicamente como fechas de calendario. La hora de la cita
se conserva separadamente como texto HH:MM:SS.

#### Conversión de fechas y horas

`fecha_bson` convierte las fechas de MySQL a una representación
compatible con BSON, el formato utilizado por MongoDB.

`hora_texto` convierte la hora de cada cita al formato `HH:MM:SS`.

Las fechas a medianoche UTC representan días del calendario, no
instantes de atención. La hora se conserva en un campo separado.

In [11]:
from datetime import datetime, time, timedelta, timezone
from bson import BSON

def fecha_bson(valor):
    """Representar una fecha de calendario a medianoche UTC."""
    if valor is None:
        return None

    return datetime(
        valor.year, valor.month, valor.day,
        tzinfo=timezone.utc
    )

def hora_texto(valor):
    """Conservar la hora SQL como HH:MM:SS."""
    if isinstance(valor, timedelta):
        segundos = valor.total_seconds()

        if not segundos.is_integer() or not 0 <= segundos < 86400:
            raise ValueError(f"Hora de cita fuera del formato esperado: {valor}")

        horas, resto = divmod(int(segundos), 3600)
        minutos, segundos = divmod(resto, 60)
        return f"{horas:02d}:{minutos:02d}:{segundos:02d}"

    if isinstance(valor, time):
        if valor.microsecond:
            raise ValueError("La hora contiene fracciones de segundo.")
        return valor.strftime("%H:%M:%S")

    raise TypeError(f"Tipo de hora no reconocido: {type(valor)}")

#### Preparación de las nueve colecciones principales

Se construye un diccionario llamado `documentos`, que contiene una lista
de documentos para cada colección.

Los campos de MySQL se adaptan a la estructura definida en el modelo
documental. Se conservan los identificadores originales y se incorporan
las relaciones agrupadas en la etapa anterior.

Al finalizar, se comprueba que los identificadores sean únicos dentro
de cada colección y que todos los documentos puedan codificarse en BSON.

Los documentos todavía están en memoria; la inserción se realiza después.

In [12]:
# Documentos preparados en memoria, separados por colección.
documentos = {}

documentos["Centros"] = [
    {
        "_id": fila["Codigo_Centro"],
        "nombre": fila["Nombre_Centro"],
        "comuna": fila["Comuna_Centro"],
        "region": fila["Region_Centro"],
        "medicos_rut": [
            relacion["RUT_Medico"]
            for relacion in datos_mysql["centro_medico_medico"]
            if relacion["Codigo_Centro"] == fila["Codigo_Centro"]
        ]
    }
    for fila in datos_mysql["centro_medico"]
]

documentos["Especialidades"] = [
    {
        "_id": fila["ID_Especialidad"],
        "nombre": fila["Nombre_Especialidad"]
    }
    for fila in datos_mysql["especialidad"]
]

documentos["Previsiones"] = [
    {
        "_id": fila["ID_Prevision"],
        "tipo": fila["Prevision_Paciente"]
    }
    for fila in datos_mysql["prevision"]
]

documentos["Estados"] = [
    {
        "_id": fila["ID_Estado"],
        "nombre": fila["Estado_Cita"]
    }
    for fila in datos_mysql["estado"]
]

documentos["Diagnosticos"] = [
    {
        "_id": fila["Codigo_Diagnostico"],
        "descripcion": fila["Descripcion_Diagnostico"]
    }
    for fila in datos_mysql["diagnostico"]
]

documentos["Pacientes"] = [
    {
        "_id": fila["RUT_Paciente"],
        "nombre": fila["Nombre_Paciente"],
        "telefono": fila["Telefono_Paciente"],
        "nacimiento": fecha_bson(fila["Fecha-Nacimiento_Paciente"]),
        "sexo": fila["Sexo_Paciente"],
        "prevision_id": fila["ID_Prevision"],
        "email": fila["Email_Paciente"],
        "contrasena_hash": fila["Contrasena_Hash"]
    }
    for fila in datos_mysql["paciente"]
]

documentos["Medicos"] = [
    {
        "_id": fila["RUT_Medico"],
        "nombre": fila["Nombre_Medico"],
        "email": fila["E-mail_Medico"],
        "contrasena_hash": fila["Contrasena_Hash"],
        "centro_ids": list(centros_por_medico.get(fila["RUT_Medico"], [])),
        "especialidad_ids": list(
            especialidades_por_medico.get(fila["RUT_Medico"], [])
        )
    }
    for fila in datos_mysql["medico"]
]

documentos["Citas"] = [
    {
        "_id": fila["Nro_Cita"],
        "fecha": fecha_bson(fila["Fecha_Cita"]),
        "hora": hora_texto(fila["Hora_Cita"]),
        "estado_id": fila["ID_Estado"],
        "paciente_rut": fila["RUT_Paciente"],
        "medico_rut": fila["RUT_Medico"],
        "especialidad_id": fila["ID_Especialidad"],
        "centro_id": fila["Codigo_Centro"]
    }
    for fila in datos_mysql["cita"]
]

documentos["Atenciones"] = [
    {
        "_id": fila["Nro_Atencion"],
        "cita_id": fila["Nro_Cita"],
        "motivo": fila["Motivo_Atencion"],
        "observacion": fila["Observaciones_Atencion"],
        "diagnosticos_id": list(
            diagnosticos_por_atencion.get(fila["Nro_Atencion"], [])
        ),
        "recetas": list(
            recetas_por_atencion.get(fila["Nro_Atencion"], [])
        )
    }
    for fila in datos_mysql["atencion"]
]

# Verificar identificadores únicos y compatibilidad con BSON.
for coleccion, lista in documentos.items():
    ids = [documento["_id"] for documento in lista]

    assert len(ids) == len(set(ids)), (
        f"Hay identificadores duplicados en {coleccion}"
    )

    for documento in lista:
        BSON.encode(documento)

    print(f"{coleccion}: {len(lista)} documentos preparados")

Centros: 5 documentos preparados
Especialidades: 8 documentos preparados
Previsiones: 3 documentos preparados
Estados: 5 documentos preparados
Diagnosticos: 14 documentos preparados
Pacientes: 49 documentos preparados
Medicos: 22 documentos preparados
Citas: 205 documentos preparados
Atenciones: 60 documentos preparados


#### Validación de referencias y asociaciones

Se comprueba que los identificadores referenciados existan entre los
documentos preparados.

También se verifica que cada médico esté asociado al centro y a la
especialidad de sus citas, y que no existan varias atenciones para
una misma cita.

Estas son comprobaciones realizadas por el programa de migración.
No crean restricciones de claves foráneas dentro de MongoDB.

In [13]:
ids = {
    nombre: {doc["_id"] for doc in lista}
    for nombre, lista in documentos.items()
}

medicos_por_id = {
    doc["_id"]: doc for doc in documentos["Medicos"]
}

for paciente in documentos["Pacientes"]:
    assert paciente["prevision_id"] in ids["Previsiones"]

for medico in documentos["Medicos"]:
    assert set(medico["centro_ids"]) <= ids["Centros"]
    assert set(medico["especialidad_ids"]) <= ids["Especialidades"]

for cita in documentos["Citas"]:
    assert cita["medico_rut"] in ids["Medicos"]
    assert cita["centro_id"] in ids["Centros"]
    assert cita["especialidad_id"] in ids["Especialidades"]
    assert cita["estado_id"] in ids["Estados"]

    # El origen permite citas cuyo paciente sea NULL.
    if cita["paciente_rut"] is not None:
        assert cita["paciente_rut"] in ids["Pacientes"]

    medico = medicos_por_id[cita["medico_rut"]]
    assert cita["centro_id"] in medico["centro_ids"], (
        f"Cita {cita['_id']}: médico no asociado al centro"
    )
    assert cita["especialidad_id"] in medico["especialidad_ids"], (
        f"Cita {cita['_id']}: especialidad no asignada al médico"
    )

for atencion in documentos["Atenciones"]:
    assert atencion["cita_id"] in ids["Citas"]
    assert set(atencion["diagnosticos_id"]) <= ids["Diagnosticos"]

# Comprobar que no quedaron recetas o asociaciones fuera de los documentos.
assert set(recetas_por_atencion) <= ids["Atenciones"]
assert set(diagnosticos_por_atencion) <= ids["Atenciones"]
assert set(centros_por_medico) <= ids["Medicos"]
assert set(especialidades_por_medico) <= ids["Medicos"]

citas_atendidas = [doc["cita_id"] for doc in documentos["Atenciones"]]
assert len(citas_atendidas) == len(set(citas_atendidas)), (
    "Existen varias atenciones asociadas a una misma cita."
)

print("Referencias y asociaciones verificadas correctamente.")

Referencias y asociaciones verificadas correctamente.


### Inserción en MongoDB

Se insertan los documentos preparados en las colecciones principales
de salud_usm, utilizando la conexión con write concern majority.

Antes de insertar, se comprueba que las colecciones de destino estén
vacías para evitar duplicar datos o sobrescribir información existente.

La siguiente celda modifica MongoDB. La carga se realiza por colección
y no constituye una única transacción para toda la migración.

Si ocurre un error durante la inserción, puede existir una carga parcial
que debe revisarse antes de intentar nuevamente el procedimiento.

### Carga de las colecciones principales

In [14]:
# Comprobar todos los destinos antes de comenzar la carga.
ocupadas = [
    nombre
    for nombre in documentos
    if db_mongo[nombre].count_documents({}) > 0
]

if ocupadas:
    raise RuntimeError(
        f"Estas colecciones ya contienen datos: {ocupadas}. "
        "No se ha iniciado una nueva carga."
    )

# Insertar los documentos de cada colección.
for nombre, lista in documentos.items():
    if lista:
        resultado = db_mongo[nombre].insert_many(lista)
        print(f"{nombre}: {len(resultado.inserted_ids)} insertados")

print("\nCarga de las nueve colecciones principales completada.")

Centros: 5 insertados
Especialidades: 8 insertados
Previsiones: 3 insertados
Estados: 5 insertados
Diagnosticos: 14 insertados
Pacientes: 49 insertados
Medicos: 22 insertados
Citas: 205 insertados
Atenciones: 60 insertados

Carga de las nueve colecciones principales completada.


### 4.2.2 Evidencia de conservación de los datos

Se comparan los resultados de SELECT COUNT(*) obtenidos desde MySQL
con los conteos de documentos consultados directamente en MongoDB
mediante count_documents({}).

Las recetas y las relaciones que se incorporaron como arreglos se
verifican contando sus elementos mediante agregaciones.

Adicionalmente, se compara el contenido recuperado desde MongoDB
con los documentos preparados durante la transformación.

### Comparación de conteos entre tablas y colecciones

In [15]:
from IPython.display import display, Markdown

# Correspondencia entre las tablas originales y las colecciones.
equivalencias = {
    "centro_medico": "Centros",
    "especialidad": "Especialidades",
    "prevision": "Previsiones",
    "estado": "Estados",
    "diagnostico": "Diagnosticos",
    "paciente": "Pacientes",
    "medico": "Medicos",
    "cita": "Citas",
    "atencion": "Atenciones"
}

resultados_conteos = []

for tabla, coleccion in equivalencias.items():
    total_mysql = conteos_mysql[tabla]

    # Consultar los documentos realmente almacenados.
    total_mongo = db_mongo[coleccion].count_documents({})

    resultados_conteos.append({
        "tabla": tabla,
        "coleccion": coleccion,
        "mysql": total_mysql,
        "mongodb": total_mongo,
        "coincide": total_mysql == total_mongo
    })

# Mostrar ambos conteos y la consulta SQL equivalente.
filas = [
    "| Consulta MySQL | Colección | MySQL | MongoDB | Resultado |",
    "|---|---|---:|---:|---|"
]

for resultado in resultados_conteos:
    estado = "Correcto" if resultado["coincide"] else "REVISAR"
    filas.append(
        f"| `SELECT COUNT(*) FROM {resultado['tabla']};` "
        f"| {resultado['coleccion']} "
        f"| {resultado['mysql']} "
        f"| {resultado['mongodb']} "
        f"| {estado} |"
    )

display(Markdown("\n".join(filas)))

assert all(r["coincide"] for r in resultados_conteos), (
    "Hay diferencias entre los conteos de MySQL y MongoDB."
)

| Consulta MySQL | Colección | MySQL | MongoDB | Resultado |
|---|---|---:|---:|---|
| `SELECT COUNT(*) FROM centro_medico;` | Centros | 5 | 5 | Correcto |
| `SELECT COUNT(*) FROM especialidad;` | Especialidades | 8 | 8 | Correcto |
| `SELECT COUNT(*) FROM prevision;` | Previsiones | 3 | 3 | Correcto |
| `SELECT COUNT(*) FROM estado;` | Estados | 5 | 5 | Correcto |
| `SELECT COUNT(*) FROM diagnostico;` | Diagnosticos | 14 | 14 | Correcto |
| `SELECT COUNT(*) FROM paciente;` | Pacientes | 49 | 49 | Correcto |
| `SELECT COUNT(*) FROM medico;` | Medicos | 22 | 22 | Correcto |
| `SELECT COUNT(*) FROM cita;` | Citas | 205 | 205 | Correcto |
| `SELECT COUNT(*) FROM atencion;` | Atenciones | 60 | 60 | Correcto |

#### Conteo de recetas y relaciones incorporadas en arreglos

Las recetas y algunas relaciones ya no tienen una colección independiente.
Por eso, se cuentan los elementos de sus arreglos mediante agregaciones
en MongoDB y se comparan con los conteos de las tablas originales.

La relación médico–centro se verifica en ambas representaciones:
`Medicos.centro_ids` y `Centros.medicos_rut`.

In [16]:
def contar_elementos(coleccion, campo):
    """Sumar en MongoDB los elementos de un arreglo."""
    resultado = list(db_mongo[coleccion].aggregate([
        {
            "$group": {
                "_id": None,
                "total": {"$sum": {"$size": f"${campo}"}}
            }
        }
    ]))

    return resultado[0]["total"] if resultado else 0


relaciones = [
    ("receta", "Atenciones", "recetas"),
    ("atencion_diagnostico", "Atenciones", "diagnosticos_id"),
    ("medico_especialidad", "Medicos", "especialidad_ids"),
    ("centro_medico_medico", "Medicos", "centro_ids"),
    # La misma relación también está representada en Centros.
    ("centro_medico_medico", "Centros", "medicos_rut")
]

comparaciones_relaciones = []

for tabla, coleccion, campo in relaciones:
    origen = conteos_mysql[tabla]
    destino = contar_elementos(coleccion, campo)

    comparaciones_relaciones.append(origen == destino)

    print(
        f"{tabla} → {coleccion}.{campo}: "
        f"MySQL = {origen}, MongoDB = {destino}"
    )

assert all(comparaciones_relaciones), (
    "Hay diferencias en las recetas o en las relaciones migradas."
)

print("\nTodos los conteos de recetas y relaciones coinciden.")

receta → Atenciones.recetas: MySQL = 80, MongoDB = 80
atencion_diagnostico → Atenciones.diagnosticos_id: MySQL = 72, MongoDB = 72
medico_especialidad → Medicos.especialidad_ids: MySQL = 59, MongoDB = 59
centro_medico_medico → Medicos.centro_ids: MySQL = 110, MongoDB = 110
centro_medico_medico → Centros.medicos_rut: MySQL = 110, MongoDB = 110

Todos los conteos de recetas y relaciones coinciden.


#### Comparación del contenido de los documentos

Se recuperan los documentos almacenados en MongoDB y se comparan con
los preparados durante la transformación.

La comparación identifica documentos faltantes, adicionales o con
contenido diferente. Antes de comparar, se normaliza su representación
BSON para tratar las fechas de manera consistente.

Esta comprobación valida que la escritura conserve los documentos
preparados; complementa las verificaciones de conteos y relaciones.

In [17]:
from bson import BSON
from bson.codec_options import CodecOptions

# Normalizar fechas y otros tipos tal como se almacenan en BSON.
opciones_bson = CodecOptions(tz_aware=True)

def normalizar_documento(documento):
    return BSON(BSON.encode(documento)).decode(
        codec_options=opciones_bson
    )

for nombre, preparados in documentos.items():
    esperados = {
        doc["_id"]: normalizar_documento(doc)
        for doc in preparados
    }

    recuperados = {
        doc["_id"]: normalizar_documento(doc)
        for doc in db_mongo[nombre].find({})
    }

    faltantes = set(esperados) - set(recuperados)
    adicionales = set(recuperados) - set(esperados)

    distintos = [
        identificador
        for identificador in set(esperados) & set(recuperados)
        if esperados[identificador] != recuperados[identificador]
    ]

    print(
        f"{nombre}: faltantes={len(faltantes)}, "
        f"adicionales={len(adicionales)}, "
        f"contenido diferente={len(distintos)}"
    )

    assert not faltantes and not adicionales and not distintos, (
        f"Se encontraron diferencias en {nombre}."
    )

print("\nEl contenido almacenado coincide con los documentos preparados.")

Centros: faltantes=0, adicionales=0, contenido diferente=0
Especialidades: faltantes=0, adicionales=0, contenido diferente=0
Previsiones: faltantes=0, adicionales=0, contenido diferente=0
Estados: faltantes=0, adicionales=0, contenido diferente=0
Diagnosticos: faltantes=0, adicionales=0, contenido diferente=0
Pacientes: faltantes=0, adicionales=0, contenido diferente=0
Medicos: faltantes=0, adicionales=0, contenido diferente=0
Citas: faltantes=0, adicionales=0, contenido diferente=0
Atenciones: faltantes=0, adicionales=0, contenido diferente=0

El contenido almacenado coincide con los documentos preparados.


#### Verificación de los mínimos de población

Se comprueba que MongoDB contenga las cantidades mínimas solicitadas
de centros, especialidades, previsiones, pacientes, médicos, citas,
atenciones y recetas.

Además, se verifica que existan centros en al menos dos regiones
y que ninguna atención carezca de diagnóstico.

In [18]:
minimos = {
    "Centros": 4,
    "Especialidades": 8,
    "Previsiones": 3,
    "Pacientes": 40,
    "Medicos": 15,
    "Citas": 120,
    "Atenciones": 60
}

for coleccion, minimo in minimos.items():
    total = db_mongo[coleccion].count_documents({})
    print(f"{coleccion}: {total} / mínimo {minimo}")
    assert total >= minimo

regiones = db_mongo["Centros"].distinct("region")
assert len(regiones) >= 2
print("Regiones distintas:", len(regiones))

total_recetas = contar_elementos("Atenciones", "recetas")
assert total_recetas >= 80
print("Recetas embebidas:", total_recetas)

atenciones_sin_diagnostico = db_mongo["Atenciones"].count_documents({
    "diagnosticos_id.0": {"$exists": False}
})
assert atenciones_sin_diagnostico == 0
print("Atenciones sin diagnóstico:", atenciones_sin_diagnostico)

print("\nLos mínimos de población se cumplen en MongoDB.")

Centros: 5 / mínimo 4
Especialidades: 8 / mínimo 8
Previsiones: 3 / mínimo 3
Pacientes: 49 / mínimo 40
Medicos: 22 / mínimo 15
Citas: 205 / mínimo 120
Atenciones: 60 / mínimo 60
Regiones distintas: 3
Recetas embebidas: 80
Atenciones sin diagnóstico: 0

Los mínimos de población se cumplen en MongoDB.


### Generación de Comprobantes y Fichas

Se genera un documento de Comprobantes por cada cita y un documento
de Fichas por cada atención.

Ambas colecciones reúnen información de las colecciones principales
para facilitar la presentación de los documentos al usuario.
Se conserva el identificador de la cita o atención en el campo _id.

Los documentos representan una copia de la información disponible
en el momento indicado por generado_en. No se actualizan automáticamente
cuando cambian las colecciones principales.

Se incluyen las citas de todos los estados. Si una cita no tiene
paciente asociado, su campo paciente se conserva como null.

#### Lectura de los datos para generar comprobantes y fichas

Se consultan las nueve colecciones principales y se organizan sus
documentos por identificador para relacionarlos durante la generación.

También se registra `generado_en`, que identifica el momento de
preparación de los documentos derivados.

Durante esta etapa no deben realizarse modificaciones simultáneas
sobre los datos de origen.

In [19]:
from datetime import datetime, timezone
from copy import deepcopy

# Ejecutar esta lectura sin modificaciones simultáneas en la base.
# Este bloque no bloquea las colecciones ni impide escrituras externas.
fuentes = [
    "Centros",
    "Especialidades",
    "Previsiones",
    "Estados",
    "Diagnosticos",
    "Pacientes",
    "Medicos",
    "Citas",
    "Atenciones"
]

# Organizar los documentos por su identificador para relacionarlos.
origen_mongo = {
    nombre: {
        documento["_id"]: documento
        for documento in db_mongo[nombre].find({})
    }
    for nombre in fuentes
}

generado_en = datetime.now(timezone.utc)

print("Datos de las colecciones principales leídos correctamente.")

Datos de las colecciones principales leídos correctamente.


#### Construcción de los comprobantes

Se prepara un comprobante por cada cita, reuniendo su fecha, hora,
estado, paciente, previsión, médico, especialidad y centro.

El identificador del comprobante coincide con el de la cita.
Si la cita no tiene paciente asociado, se conserva ese dato como `null`.

In [20]:
comprobantes = []

for cita in origen_mongo["Citas"].values():
    medico = origen_mongo["Medicos"][cita["medico_rut"]]
    centro = origen_mongo["Centros"][cita["centro_id"]]
    especialidad = origen_mongo["Especialidades"][cita["especialidad_id"]]
    estado = origen_mongo["Estados"][cita["estado_id"]]

    paciente_documento = None

    if cita["paciente_rut"] is not None:
        paciente = origen_mongo["Pacientes"][cita["paciente_rut"]]
        prevision = origen_mongo["Previsiones"][paciente["prevision_id"]]

        paciente_documento = {
            "rut": paciente["_id"],
            "nombre": paciente["nombre"],
            "telefono": paciente["telefono"],
            "prevision": {
                "id": prevision["_id"],
                "tipo": prevision["tipo"]
            }
        }

    comprobantes.append({
        "_id": cita["_id"],
        "generado_en": generado_en,
        "cita": {
            "id": cita["_id"],
            "fecha": cita["fecha"],
            "hora": cita["hora"],
            "estado": {
                "id": estado["_id"],
                "nombre": estado["nombre"]
            },
            "paciente": paciente_documento,
            "medico": {
                "rut": medico["_id"],
                "nombre": medico["nombre"],
                "especialidad": {
                    "id": especialidad["_id"],
                    "nombre": especialidad["nombre"]
                }
            },
            "centro": {
                "id": centro["_id"],
                "nombre": centro["nombre"],
                "comuna": centro["comuna"],
                "region": centro["region"]
            }
        }
    })

print("Comprobantes preparados:", len(comprobantes))

Comprobantes preparados: 205


#### Construcción de las fichas clínicas

Se prepara una ficha por cada atención, incorporando los datos de
la cita, paciente, médico y centro, junto con el motivo, las observaciones,
los diagnósticos y las recetas.

Los códigos de diagnóstico se acompañan de sus descripciones.
Las recetas se copian mediante `deepcopy` para que sus estructuras
en memoria sean independientes de las utilizadas como origen.

In [21]:
fichas = []

for atencion in origen_mongo["Atenciones"].values():
    cita = origen_mongo["Citas"][atencion["cita_id"]]
    medico = origen_mongo["Medicos"][cita["medico_rut"]]
    centro = origen_mongo["Centros"][cita["centro_id"]]
    especialidad = origen_mongo["Especialidades"][cita["especialidad_id"]]

    paciente_documento = None

    if cita["paciente_rut"] is not None:
        paciente = origen_mongo["Pacientes"][cita["paciente_rut"]]

        paciente_documento = {
            "rut": paciente["_id"],
            "nombre": paciente["nombre"],
            "nacimiento": paciente["nacimiento"],
            "sexo": paciente["sexo"]
        }

    diagnosticos = [
        {
            "codigo": codigo,
            "descripcion": origen_mongo["Diagnosticos"][codigo]["descripcion"]
        }
        for codigo in atencion["diagnosticos_id"]
    ]

    fichas.append({
        "_id": atencion["_id"],
        "generado_en": generado_en,
        "cita": {
            "id": cita["_id"],
            "fecha": cita["fecha"],
            "hora": cita["hora"],
            "centro": {
                "id": centro["_id"],
                "nombre": centro["nombre"]
            },
            "paciente": paciente_documento,
            "medico": {
                "rut": medico["_id"],
                "nombre": medico["nombre"],
                "email": medico["email"],
                "especialidad": {
                    "id": especialidad["_id"],
                    "nombre": especialidad["nombre"]
                }
            },
            "atencion": {
                "id": atencion["_id"],
                "motivo": atencion["motivo"],
                "observacion": atencion["observacion"],
                "diagnosticos": diagnosticos,
                "recetas": deepcopy(atencion["recetas"])
            }
        }
    })

print("Fichas preparadas:", len(fichas))

Fichas preparadas: 60


#### Validación e inserción de comprobantes y fichas

In [22]:
from bson import BSON

documentos_derivados = {
    "Comprobantes": comprobantes,
    "Fichas": fichas
}

# Verificar que cada cita y atención tenga su documento correspondiente.
assert {d["_id"] for d in comprobantes} == set(origen_mongo["Citas"])
assert {d["_id"] for d in fichas} == set(origen_mongo["Atenciones"])

# Verificar identificadores únicos y compatibilidad BSON.
for nombre, lista in documentos_derivados.items():
    assert len(lista) == len({d["_id"] for d in lista})

    for documento in lista:
        BSON.encode(documento)

# Revisar ambos destinos antes de comenzar.
ocupadas = [
    nombre
    for nombre in documentos_derivados
    if db_mongo[nombre].count_documents({}) > 0
]

if ocupadas:
    raise RuntimeError(
        f"Ya contienen datos: {ocupadas}. No se inició una nueva carga."
    )

for nombre, lista in documentos_derivados.items():
    if lista:
        resultado = db_mongo[nombre].insert_many(lista)
        print(f"{nombre}: {len(resultado.inserted_ids)} insertados")

Comprobantes: 205 insertados
Fichas: 60 insertados


#### Comparación de conteos y contenido de los documentos derivados

In [23]:
from bson.codec_options import CodecOptions

opciones = CodecOptions(tz_aware=True)

def normalizar_bson(documento):
    return BSON(BSON.encode(documento)).decode(codec_options=opciones)

for tabla, coleccion in [
    ("cita", "Comprobantes"),
    ("atencion", "Fichas")
]:
    total_mysql = conteos_mysql[tabla]
    total_mongo = db_mongo[coleccion].count_documents({})

    print(
        f"SELECT COUNT(*) FROM {tabla}; → {total_mysql} | "
        f"{coleccion}.count_documents({{}}) → {total_mongo}"
    )

    assert total_mysql == total_mongo

    esperados = {
        d["_id"]: normalizar_bson(d)
        for d in documentos_derivados[coleccion]
    }

    almacenados = {
        d["_id"]: normalizar_bson(d)
        for d in db_mongo[coleccion].find({})
    }

    assert esperados == almacenados, (
        f"El contenido almacenado difiere en {coleccion}"
    )

print("\nComprobantes y Fichas: conteos y contenido correctos.")

SELECT COUNT(*) FROM cita; → 205 | Comprobantes.count_documents({}) → 205
SELECT COUNT(*) FROM atencion; → 60 | Fichas.count_documents({}) → 60

Comprobantes y Fichas: conteos y contenido correctos.


#### Visualización de un comprobante y una ficha almacenados

In [24]:
from bson import json_util

for nombre in ["Comprobantes", "Fichas"]:
    ejemplo = db_mongo[nombre].find_one(sort=[("_id", 1)])

    print(f"\nEjemplo de {nombre}:")
    print(json_util.dumps(
        ejemplo,
        indent=2,
        ensure_ascii=False
    ))


Ejemplo de Comprobantes:
{
  "_id": 1,
  "generado_en": {
    "$date": "2026-11-05T02:53:18.401Z"
  },
  "cita": {
    "id": 1,
    "fecha": {
      "$date": "2026-09-21T00:00:00Z"
    },
    "hora": "09:00:00",
    "estado": {
      "id": 2,
      "nombre": "Atendida"
    },
    "paciente": {
      "rut": "00000000-0",
      "nombre": "Martina Díaz",
      "telefono": "912345670",
      "prevision": {
        "id": 2,
        "tipo": "Isapre"
      }
    },
    "medico": {
      "rut": "00000000-1",
      "nombre": "Dr. Andrés Vidal",
      "especialidad": {
        "id": 4,
        "nombre": "Medicina General"
      }
    },
    "centro": {
      "id": 1,
      "nombre": "SaludUSM San Joaquín",
      "comuna": "San Joaquín",
      "region": "RM"
    }
  }
}

Ejemplo de Fichas:
{
  "_id": 1,
  "generado_en": {
    "$date": "2026-11-05T02:53:18.401Z"
  },
  "cita": {
    "id": 1,
    "fecha": {
      "$date": "2026-09-21T00:00:00Z"
    },
    "hora": "09:00:00",
    "centro": {
      

### 4.2.3 Verificación de datos para consultas y replicación

Los ejemplos se muestran en Extended JSON mediante `json_util`,
que permite representar tipos BSON como las fechas.

Se comprueba la existencia de datos para ejecutar las cinco consultas
de la sección 4.3 con resultados no vacíos.

También se verifica que existan citas en estado Reservada o Confirmada
con fecha anterior al día actual en Chile.

Estas comprobaciones no modifican los datos. Los filtros y agrupaciones
se ejecutan en MongoDB.

Estas verificaciones buscan casos disponibles para los ejercicios.
No reemplazan la implementación completa de las consultas de la sección 4.3.

#### Preparación de los estados utilizados en las verificaciones

Se obtienen los identificadores desde el catálogo `Estados`, utilizando
sus nombres. También se crea el diccionario que almacenará el resultado
de cada comprobación.

In [25]:
from IPython.display import display

# Obtener los identificadores desde el catálogo, sin asumir sus números.
estados_por_nombre = {
    documento["nombre"]: documento["_id"]
    for documento in db_mongo["Estados"].find({})
}

for nombre in ["Atendida", "No Asistió", "Reservada", "Confirmada"]:
    assert nombre in estados_por_nombre, (
        f"No se encontró el estado '{nombre}' en el catálogo."
    )

id_atendida = estados_por_nombre["Atendida"]
id_no_asistio = estados_por_nombre["No Asistió"]
ids_pendientes = [
    estados_por_nombre["Reservada"],
    estados_por_nombre["Confirmada"]
]

# Guardar el resultado de cada comprobación.
verificaciones_423 = {}

#### a. Disponibilidad de una agenda diaria

Se busca una combinación de médico y fecha que tenga citas con paciente
asociado, para utilizarla como ejemplo en la consulta de agenda.

In [26]:
agenda_disponible = list(db_mongo["Citas"].aggregate([
    {"$match": {"paciente_rut": {"$ne": None}}},
    {
        "$group": {
            "_id": {
                "medico_rut": "$medico_rut",
                "fecha": "$fecha"
            },
            "cantidad_citas": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "cantidad_citas": -1,
            "_id.medico_rut": 1,
            "_id.fecha": 1
        }
    },
    {"$limit": 1}
]))

verificaciones_423["a. Agenda diaria"] = bool(agenda_disponible)

display(agenda_disponible)

[{'_id': {'medico_rut': '00000000-1',
   'fecha': datetime.datetime(2026, 11, 2, 0, 0, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC'))},
  'cantidad_citas': 3}]

#### b. Disponibilidad de un historial clínico

Se relacionan las atenciones con sus citas para encontrar un paciente
que tenga atenciones registradas.

In [27]:
historial_disponible = list(db_mongo["Atenciones"].aggregate([
    {
        "$lookup": {
            "from": "Citas",
            "localField": "cita_id",
            "foreignField": "_id",
            "as": "cita"
        }
    },
    {"$unwind": "$cita"},
    {"$match": {"cita.paciente_rut": {"$ne": None}}},
    {
        "$group": {
            "_id": "$cita.paciente_rut",
            "cantidad_atenciones": {"$sum": 1}
        }
    },
    {"$sort": {"cantidad_atenciones": -1, "_id": 1}},
    {"$limit": 1}
]))

verificaciones_423["b. Historial clínico"] = bool(historial_disponible)

display(historial_disponible)

[{'_id': '10987654-3', 'cantidad_atenciones': 4}]

#### c. Centros con inasistencia superior al 15 %

Se calcula el porcentaje de citas con estado `No Asistió` respecto
del total de citas de cada centro.

Se seleccionan los centros cuyo porcentaje supera estrictamente el 15 %.
El filtro se aplica antes de redondear el resultado a dos decimales.

In [28]:
centros_con_inasistencia = list(db_mongo["Citas"].aggregate([
    {
        "$group": {
            "_id": "$centro_id",
            "total_citas": {"$sum": 1},
            "inasistencias": {
                "$sum": {
                    "$cond": [
                        {"$eq": ["$estado_id", id_no_asistio]},
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$set": {
            "porcentaje": {
                "$multiply": [
                    {"$divide": ["$inasistencias", "$total_citas"]},
                    100
                ]
            }
        }
    },
    # Aplicar el umbral al valor sin redondear.
    {"$match": {"porcentaje": {"$gt": 15}}},
    {
        "$set": {
            "porcentaje": {"$round": ["$porcentaje", 2]}
        }
    },
    {"$sort": {"porcentaje": -1, "_id": 1}}
]))

verificaciones_423["c. Inasistencia superior al 15 %"] = bool(
    centros_con_inasistencia
)

display(centros_con_inasistencia)

[{'_id': 1, 'total_citas': 48, 'inasistencias': 8, 'porcentaje': 16.67}]

#### d. Disponibilidad de datos para el ranking de medicamentos

Se relacionan las atenciones con sus citas y se examinan las recetas
embebidas para identificar una especialidad que tenga medicamentos
registrados.

Esta comprobación selecciona un caso disponible; todavía no construye
el ranking final de los cinco medicamentos.

In [29]:
especialidades_con_recetas = list(db_mongo["Atenciones"].aggregate([
    {
        "$lookup": {
            "from": "Citas",
            "localField": "cita_id",
            "foreignField": "_id",
            "as": "cita"
        }
    },
    {"$unwind": "$cita"},
    {"$unwind": "$recetas"},
    {
        "$group": {
            "_id": "$cita.especialidad_id",
            "cantidad_recetas": {"$sum": 1},
            "medicamentos": {"$addToSet": "$recetas.medicamento"}
        }
    },
    {
        "$project": {
            "cantidad_recetas": 1,
            "medicamentos_distintos": {"$size": "$medicamentos"}
        }
    },
    {
        "$sort": {
            "medicamentos_distintos": -1,
            "cantidad_recetas": -1,
            "_id": 1
        }
    },
    {"$limit": 1}
]))

verificaciones_423["d. Ranking de medicamentos"] = bool(
    especialidades_con_recetas
)

display(especialidades_con_recetas)

[{'_id': 4, 'cantidad_recetas': 20, 'medicamentos_distintos': 11}]

#### e. Pacientes atendidos en al menos tres especialidades

Se consideran las citas con estado `Atendida` y se agrupan por paciente.
Se cuentan las especialidades distintas, evitando contar varias veces
una misma especialidad.

In [30]:
pacientes_multiespecialidad = list(db_mongo["Citas"].aggregate([
    {
        "$match": {
            "estado_id": id_atendida,
            "paciente_rut": {"$ne": None}
        }
    },
    {
        "$group": {
            "_id": "$paciente_rut",
            "especialidades": {"$addToSet": "$especialidad_id"}
        }
    },
    {
        "$set": {
            "cantidad_especialidades": {"$size": "$especialidades"}
        }
    },
    {"$match": {"cantidad_especialidades": {"$gte": 3}}},
    {"$sort": {"cantidad_especialidades": -1, "_id": 1}}
]))

verificaciones_423["e. Pacientes multiespecialidad"] = bool(
    pacientes_multiespecialidad
)

display(pacientes_multiespecialidad)

[{'_id': '10987654-3',
  'especialidades': [6, 1, 4, 8],
  'cantidad_especialidades': 4}]

#### Disponibilidad de citas vencidas pendientes

Se buscan citas Reservadas o Confirmadas cuya fecha sea anterior
al día actual en Chile.

La fecha de la cita se interpreta como fecha de calendario.
Esta comprobación solo consulta los datos: no cancela las citas.

In [31]:
citas_vencidas = list(db_mongo["Citas"].aggregate([
    {"$match": {"estado_id": {"$in": ids_pendientes}}},
    {
        "$match": {
            "$expr": {
                "$lt": [
                    {
                        "$dateToString": {
                            "date": "$fecha",
                            "format": "%Y-%m-%d",
                            "timezone": "UTC"
                        }
                    },
                    {
                        "$dateToString": {
                            "date": "$$NOW",
                            "format": "%Y-%m-%d",
                            "timezone": "America/Santiago"
                        }
                    }
                ]
            }
        }
    },
    {
        "$group": {
            "_id": "$estado_id",
            "cantidad": {"$sum": 1}
        }
    },
    {"$sort": {"_id": 1}}
]))

verificaciones_423["Citas vencidas Reservadas o Confirmadas"] = bool(
    citas_vencidas
)

display(citas_vencidas)

[{'_id': 0, 'cantidad': 4}, {'_id': 1, 'cantidad': 2}]

#### Resumen de las verificaciones

Se muestra `CUMPLE` o `FALTAN DATOS` para cada condición evaluada.
El resumen utiliza los resultados de las celdas anteriores, por lo que
estas deben ejecutarse antes de generar nuevamente esta salida.

In [32]:
for requisito, cumple in verificaciones_423.items():
    resultado = "CUMPLE" if cumple else "FALTAN DATOS"
    print(f"{requisito}: {resultado}")

if all(verificaciones_423.values()):
    print(
        "\nSe encontraron datos para los cinco casos de consulta "
        "y para la actualización de citas vencidas."
    )
else:
    print(
        "\nHay condiciones pendientes. Revisar los resultados "
        "antes de modificar o agregar datos."
    )

a. Agenda diaria: CUMPLE
b. Historial clínico: CUMPLE
c. Inasistencia superior al 15 %: CUMPLE
d. Ranking de medicamentos: CUMPLE
e. Pacientes multiespecialidad: CUMPLE
Citas vencidas Reservadas o Confirmadas: CUMPLE

Se encontraron datos para los cinco casos de consulta y para la actualización de citas vencidas.


# 4.3. Consultas

Las consultas se resuelven en MongoDB mediante `aggregate()`. Python solo
entrega los parámetros y muestra los resultados.

Se consultan las colecciones `Citas` y `Atenciones`, que son la fuente de
verdad del sistema. `Comprobantes` y `Fichas` contienen copias para presentar
las vistas de usuario, por lo que no se utilizan para responder estas consultas.

Cada consulta se acompaña de su equivalente SQL, que también se ejecuta sobre
la base MySQL de la Tarea 2 para comparar ambos resultados. Esta sección debe
ejecutarse antes de la 4.4, porque esa sección modifica los datos de MongoDB.

#### Funciones para mostrar y comparar resultados

Estas funciones no participan en la resolución de las consultas.

In [33]:
from decimal import Decimal
from IPython.display import display, Markdown

def mostrar_tabla(filas):
    """Mostrar una lista de documentos como tabla."""
    if not filas:
        print("Sin resultados.")
        return

    columnas = list(filas[0])
    lineas = [
        "| " + " | ".join(columnas) + " |",
        "|" + "---|" * len(columnas)
    ]

    for fila in filas:
        valores = [
            ", ".join(fila[c]) if isinstance(fila[c], list) else str(fila[c])
            for c in columnas
        ]
        lineas.append("| " + " | ".join(valores) + " |")

    display(Markdown("\n".join(lineas)))


def normalizar_fila(fila):
    """Unificar decimales y listas para comparar MongoDB con MySQL."""
    normalizada = {}

    for campo, valor in fila.items():
        if isinstance(valor, Decimal):
            valor = float(valor)
        elif isinstance(valor, list):
            valor = ", ".join(valor)
        normalizada[campo] = valor

    return normalizada


def comparar_con_mysql(filas_mongo, sql, parametros=()):
    """Ejecutar la consulta SQL y comparar sus filas con las de MongoDB."""
    with conexion.cursor() as cursor:
        cursor.execute(sql, parametros)
        filas_mysql = cursor.fetchall()

    assert [normalizar_fila(f) for f in filas_mongo] == [
        normalizar_fila(f) for f in filas_mysql
    ], "MongoDB y MySQL retornan resultados distintos."

    print(f"MySQL y MongoDB coinciden: {len(filas_mysql)} filas.")

## a. Agenda diaria de un médico

Se filtran las citas por RUT del médico y fecha, y se ordenan por hora. Luego,
`$lookup` incorpora el paciente, su previsión, el centro, la especialidad y el
estado. Las referencias equivalen a los `JOIN` de la consulta SQL.

La función recibe los parámetros para reutilizarla en la prueba de alta
disponibilidad de la sección 4.4. Se conserva una cita sin paciente, si existe,
ya que el modelo relacional permite ese valor (`LEFT JOIN`).

In [34]:
from datetime import datetime, timezone

def agenda_medico(rut_medico, fecha):
    return list(db_mongo["Citas"].aggregate([
        {"$match": {"medico_rut": rut_medico, "fecha": fecha}},
        {"$sort": {"hora": 1}},
        {
            "$lookup": {
                "from": "Pacientes",
                "localField": "paciente_rut",
                "foreignField": "_id",
                "as": "paciente"
            }
        },
        {"$unwind": {"path": "$paciente", "preserveNullAndEmptyArrays": True}},
        {
            "$lookup": {
                "from": "Previsiones",
                "localField": "paciente.prevision_id",
                "foreignField": "_id",
                "as": "prevision"
            }
        },
        {"$unwind": {"path": "$prevision", "preserveNullAndEmptyArrays": True}},
        {
            "$lookup": {
                "from": "Centros",
                "localField": "centro_id",
                "foreignField": "_id",
                "as": "centro"
            }
        },
        {"$unwind": "$centro"},
        {
            "$lookup": {
                "from": "Especialidades",
                "localField": "especialidad_id",
                "foreignField": "_id",
                "as": "especialidad"
            }
        },
        {"$unwind": "$especialidad"},
        {
            "$lookup": {
                "from": "Estados",
                "localField": "estado_id",
                "foreignField": "_id",
                "as": "estado"
            }
        },
        {"$unwind": "$estado"},
        {
            "$project": {
                "_id": 0,
                "hora": 1,
                "paciente": "$paciente.nombre",
                "prevision": "$prevision.tipo",
                "centro": "$centro.nombre",
                "especialidad": "$especialidad.nombre",
                "estado": "$estado.nombre"
            }
        }
    ]))


rut_medico_agenda = "00000000-1"
fecha_agenda = datetime(2026, 11, 2, tzinfo=timezone.utc)

resultado_a = agenda_medico(rut_medico_agenda, fecha_agenda)
mostrar_tabla(resultado_a)

sql_a = """
SELECT CAST(c.Hora_Cita AS CHAR) AS hora,
       p.Nombre_Paciente AS paciente,
       pr.Prevision_Paciente AS prevision,
       cm.Nombre_Centro AS centro,
       esp.Nombre_Especialidad AS especialidad,
       e.Estado_Cita AS estado
FROM cita AS c
LEFT JOIN paciente AS p ON p.RUT_Paciente = c.RUT_Paciente
LEFT JOIN prevision AS pr ON pr.ID_Prevision = p.ID_Prevision
INNER JOIN centro_medico AS cm ON cm.Codigo_Centro = c.Codigo_Centro
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
WHERE c.RUT_Medico = %s
  AND c.Fecha_Cita = %s
ORDER BY c.Hora_Cita
"""

print(sql_a)
comparar_con_mysql(resultado_a, sql_a, (rut_medico_agenda, "2026-11-02"))

| hora | paciente | prevision | centro | especialidad | estado |
|---|---|---|---|---|---|
| 09:00:00 | Martina Díaz | Isapre | SaludUSM San Joaquín | Medicina General | No Asistió |
| 10:30:00 | Jorge Herrera | Particular | SaludUSM Vitacura | Cardiología | Reservada |
| 12:00:00 | Carlos Silva | Fonasa | SaludUSM San Joaquín | Medicina General | No Asistió |


SELECT CAST(c.Hora_Cita AS CHAR) AS hora,
       p.Nombre_Paciente AS paciente,
       pr.Prevision_Paciente AS prevision,
       cm.Nombre_Centro AS centro,
       esp.Nombre_Especialidad AS especialidad,
       e.Estado_Cita AS estado
FROM cita AS c
LEFT JOIN paciente AS p ON p.RUT_Paciente = c.RUT_Paciente
LEFT JOIN prevision AS pr ON pr.ID_Prevision = p.ID_Prevision
INNER JOIN centro_medico AS cm ON cm.Codigo_Centro = c.Codigo_Centro
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
WHERE c.RUT_Medico = %s
  AND c.Fecha_Cita = %s
ORDER BY c.Hora_Cita

MySQL y MongoDB coinciden: 3 filas.


## b. Historial clínico de un paciente

Se parte de las citas del paciente y se incorpora su atención. Las citas sin
atención se descartan, de la misma forma que con un `INNER JOIN`. Las recetas
ya están embebidas en la atención; los diagnósticos se obtienen del catálogo
`Diagnosticos` a partir de sus códigos.

El orden utiliza la fecha y, como segundo criterio, la hora de la cita.

In [35]:
def historial_paciente(rut_paciente):
    return list(db_mongo["Citas"].aggregate([
        {"$match": {"paciente_rut": rut_paciente}},
        {
            "$lookup": {
                "from": "Atenciones",
                "localField": "_id",
                "foreignField": "cita_id",
                "as": "atencion"
            }
        },
        {"$unwind": "$atencion"},
        {"$sort": {"fecha": -1, "hora": -1}},
        {
            "$lookup": {
                "from": "Centros",
                "localField": "centro_id",
                "foreignField": "_id",
                "as": "centro"
            }
        },
        {"$unwind": "$centro"},
        {
            "$lookup": {
                "from": "Especialidades",
                "localField": "especialidad_id",
                "foreignField": "_id",
                "as": "especialidad"
            }
        },
        {"$unwind": "$especialidad"},
        {
            "$lookup": {
                "from": "Medicos",
                "localField": "medico_rut",
                "foreignField": "_id",
                "as": "medico"
            }
        },
        {"$unwind": "$medico"},
        {
            "$lookup": {
                "from": "Diagnosticos",
                "localField": "atencion.diagnosticos_id",
                "foreignField": "_id",
                "as": "diagnosticos"
            }
        },
        {
            "$project": {
                "_id": 0,
                "fecha": {
                    "$dateToString": {"date": "$fecha", "format": "%Y-%m-%d"}
                },
                "centro": "$centro.nombre",
                "especialidad": "$especialidad.nombre",
                "medico": "$medico.nombre",
                "motivo": "$atencion.motivo",
                "diagnosticos": {
                    "$sortArray": {
                        "input": {
                            "$map": {
                                "input": "$diagnosticos",
                                "as": "d",
                                "in": {
                                    "codigo": "$$d._id",
                                    "descripcion": "$$d.descripcion"
                                }
                            }
                        },
                        "sortBy": {"codigo": 1}
                    }
                },
                "recetas": {
                    "$map": {
                        "input": "$atencion.recetas",
                        "as": "r",
                        "in": {
                            "medicamento": "$$r.medicamento",
                            "dosis": "$$r.dosis",
                            "dias": "$$r.dias"
                        }
                    }
                }
            }
        }
    ]))


rut_paciente_historial = "10987654-3"
resultado_b = historial_paciente(rut_paciente_historial)

for atencion in resultado_b:
    print(
        f"{atencion['fecha']} | {atencion['centro']} | "
        f"{atencion['especialidad']} | {atencion['medico']}"
    )
    print(f"  Motivo: {atencion['motivo']}")

    for diagnostico in atencion["diagnosticos"]:
        print(f"  Diagnóstico: {diagnostico['codigo']} - {diagnostico['descripcion']}")

    for receta in atencion["recetas"]:
        print(
            f"  Receta: {receta['medicamento']} | {receta['dosis']} | "
            f"{receta['dias']} días"
        )
    print()

sql_b = """
SELECT CAST(c.Fecha_Cita AS CHAR) AS fecha,
       cm.Nombre_Centro AS centro,
       esp.Nombre_Especialidad AS especialidad,
       m.Nombre_Medico AS medico,
       a.Motivo_Atencion AS motivo,
       (SELECT GROUP_CONCAT(
                   CONCAT(d.Codigo_Diagnostico, ' - ', d.Descripcion_Diagnostico)
                   ORDER BY d.Codigo_Diagnostico SEPARATOR '; ')
        FROM atencion_diagnostico AS ad
        INNER JOIN diagnostico AS d
            ON d.Codigo_Diagnostico = ad.Codigo_Diagnostico
        WHERE ad.Nro_Atencion = a.Nro_Atencion) AS diagnosticos,
       (SELECT GROUP_CONCAT(
                   CONCAT(r.Medicamento_Receta, ' | ', r.Dosis_Receta, ' | ',
                          r.`Dias-Tratamiento_Receta`, ' días')
                   ORDER BY r.ID_Receta SEPARATOR '; ')
        FROM receta AS r
        WHERE r.Nro_Atencion = a.Nro_Atencion) AS recetas
FROM atencion AS a
INNER JOIN cita AS c ON c.Nro_Cita = a.Nro_Cita
INNER JOIN centro_medico AS cm ON cm.Codigo_Centro = c.Codigo_Centro
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
INNER JOIN medico AS m ON m.RUT_Medico = c.RUT_Medico
WHERE c.RUT_Paciente = %s
ORDER BY c.Fecha_Cita DESC, c.Hora_Cita DESC
"""

print(sql_b)

# Expresar los arreglos como texto solo para comparar con GROUP_CONCAT.
resultado_b_texto = [
    {
        **atencion,
        "diagnosticos": "; ".join(
            f"{d['codigo']} - {d['descripcion']}"
            for d in atencion["diagnosticos"]
        ),
        "recetas": "; ".join(
            f"{r['medicamento']} | {r['dosis']} | {r['dias']} días"
            for r in atencion["recetas"]
        )
    }
    for atencion in resultado_b
]

comparar_con_mysql(resultado_b_texto, sql_b, (rut_paciente_historial,))

2026-10-29 | SaludUSM San Joaquín | Medicina General | Dr. Andrés Vidal
  Motivo: Motivo de prueba grupo 2
  Diagnóstico: A09 - Gastroenteritis de presunto origen infeccioso
  Receta: Medicamento de prueba grupo 4 | Pauta ficticia grupo 4 | 5 días

2026-10-26 | SaludUSM Valparaíso | Gastroenterología | Dra. Camila Fuentes
  Motivo: Diarrea y control de glicemia
  Diagnóstico: A09 - Gastroenteritis de presunto origen infeccioso
  Diagnóstico: E11 - Diabetes mellitus tipo 2
  Receta: Sales de rehidratación oral | 1 sobre después de cada deposición | 3 días
  Receta: Medicamento de prueba grupo 4 | Pauta ficticia grupo 4 | 3 días

2026-10-13 | SaludUSM San Joaquín | Cardiología | Dr. Andrés Vidal
  Motivo: Control cardiovascular
  Diagnóstico: E11 - Diabetes mellitus tipo 2
  Diagnóstico: I10 - Hipertensión esencial (primaria)
  Receta: Metformina 850 mg | 1 comprimido cada 12 horas | 90 días
  Receta: Losartán 50 mg | 1 comprimido cada 24 horas | 30 días
  Receta: Medicamento de prueba g

## c. Tasa de inasistencia por centro

Se agrupan las citas por centro para calcular su total y las citas en estado
`No Asistió`. El estado se obtiene del catálogo `Estados`.

El umbral del 15 % se aplica al porcentaje sin redondear. Por lo tanto, un
centro con 15,001 % se incluye aunque al redondear se aproxime a 15,00 %.

In [36]:
resultado_c = list(db_mongo["Citas"].aggregate([
    {
        "$lookup": {
            "from": "Estados",
            "localField": "estado_id",
            "foreignField": "_id",
            "as": "estado"
        }
    },
    {"$unwind": "$estado"},
    {
        "$group": {
            "_id": "$centro_id",
            "total_citas": {"$sum": 1},
            "inasistencias": {
                "$sum": {
                    "$cond": [{"$eq": ["$estado.nombre", "No Asistió"]}, 1, 0]
                }
            }
        }
    },
    {
        "$set": {
            "porcentaje": {
                "$multiply": [
                    {"$divide": ["$inasistencias", "$total_citas"]},
                    100
                ]
            }
        }
    },
    {"$match": {"porcentaje": {"$gt": 15}}},
    {
        "$lookup": {
            "from": "Centros",
            "localField": "_id",
            "foreignField": "_id",
            "as": "centro"
        }
    },
    {"$unwind": "$centro"},
    {
        "$project": {
            "_id": 0,
            "centro": "$centro.nombre",
            "total_citas": "$total_citas",
            "inasistencias": "$inasistencias",
            "porcentaje": {"$round": ["$porcentaje", 2]}
        }
    },
    {"$sort": {"porcentaje": -1, "centro": 1}}
]))

mostrar_tabla(resultado_c)

sql_c = """
SELECT cm.Nombre_Centro AS centro,
       COUNT(*) AS total_citas,
       SUM(e.Estado_Cita = 'No Asistió') AS inasistencias,
       ROUND(100 * SUM(e.Estado_Cita = 'No Asistió') / COUNT(*), 2) AS porcentaje
FROM cita AS c
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
INNER JOIN centro_medico AS cm ON cm.Codigo_Centro = c.Codigo_Centro
GROUP BY cm.Codigo_Centro, cm.Nombre_Centro
HAVING 100 * SUM(e.Estado_Cita = 'No Asistió') / COUNT(*) > 15
ORDER BY porcentaje DESC, centro
"""

print(sql_c)
comparar_con_mysql(resultado_c, sql_c)

| centro | total_citas | inasistencias | porcentaje |
|---|---|---|---|
| SaludUSM San Joaquín | 48 | 8 | 16.67 |


SELECT cm.Nombre_Centro AS centro,
       COUNT(*) AS total_citas,
       SUM(e.Estado_Cita = 'No Asistió') AS inasistencias,
       ROUND(100 * SUM(e.Estado_Cita = 'No Asistió') / COUNT(*), 2) AS porcentaje
FROM cita AS c
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
INNER JOIN centro_medico AS cm ON cm.Codigo_Centro = c.Codigo_Centro
GROUP BY cm.Codigo_Centro, cm.Nombre_Centro
HAVING 100 * SUM(e.Estado_Cita = 'No Asistió') / COUNT(*) > 15
ORDER BY porcentaje DESC, centro

MySQL y MongoDB coinciden: 1 filas.


## d. Ranking de medicamentos por especialidad

Se parte de la especialidad indicada y se recorren sus citas y atenciones.
`$unwind` genera un documento por cada receta embebida, y luego se cuenta la
cantidad de recetas por medicamento.

**Criterio de desempate:** los medicamentos con la misma cantidad de recetas se
ordenan alfabéticamente de forma ascendente. Este criterio se indica en el README.

In [37]:
def ranking_medicamentos(nombre_especialidad):
    return list(db_mongo["Especialidades"].aggregate([
        {"$match": {"nombre": nombre_especialidad}},
        {
            "$lookup": {
                "from": "Citas",
                "localField": "_id",
                "foreignField": "especialidad_id",
                "as": "cita"
            }
        },
        {"$unwind": "$cita"},
        {
            "$lookup": {
                "from": "Atenciones",
                "localField": "cita._id",
                "foreignField": "cita_id",
                "as": "atencion"
            }
        },
        {"$unwind": "$atencion"},
        {"$unwind": "$atencion.recetas"},
        {
            "$group": {
                "_id": "$atencion.recetas.medicamento",
                "cantidad_recetas": {"$sum": 1}
            }
        },
        {"$sort": {"cantidad_recetas": -1, "_id": 1}},
        {"$limit": 5},
        {
            "$project": {
                "_id": 0,
                "medicamento": "$_id",
                "cantidad_recetas": "$cantidad_recetas"
            }
        }
    ]))


especialidad_ranking = "Pediatría"
resultado_d = ranking_medicamentos(especialidad_ranking)
mostrar_tabla(resultado_d)

sql_d = """
SELECT r.Medicamento_Receta AS medicamento,
       COUNT(*) AS cantidad_recetas
FROM receta AS r
INNER JOIN atencion AS a ON a.Nro_Atencion = r.Nro_Atencion
INNER JOIN cita AS c ON c.Nro_Cita = a.Nro_Cita
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
WHERE esp.Nombre_Especialidad = %s
GROUP BY r.Medicamento_Receta
ORDER BY cantidad_recetas DESC, medicamento ASC
LIMIT 5
"""

print(sql_d)
comparar_con_mysql(resultado_d, sql_d, (especialidad_ranking,))

| medicamento | cantidad_recetas |
|---|---|
| Medicamento de prueba grupo 6 | 2 |
| Medicamento de prueba grupo 7 | 2 |
| Medicamento de prueba grupo 8 | 2 |
| Medicamento de prueba grupo 3 | 1 |
| Medicamento de prueba grupo 5 | 1 |


SELECT r.Medicamento_Receta AS medicamento,
       COUNT(*) AS cantidad_recetas
FROM receta AS r
INNER JOIN atencion AS a ON a.Nro_Atencion = r.Nro_Atencion
INNER JOIN cita AS c ON c.Nro_Cita = a.Nro_Cita
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
WHERE esp.Nombre_Especialidad = %s
GROUP BY r.Medicamento_Receta
ORDER BY cantidad_recetas DESC, medicamento ASC
LIMIT 5

MySQL y MongoDB coinciden: 5 filas.


## e. Pacientes multi-especialidad

Se consideran solo las citas en estado `Atendida`. Al agrupar por paciente,
`$addToSet` conserva cada especialidad una sola vez, lo que equivale a
`COUNT(DISTINCT ...)` en SQL.

Los pacientes con la misma cantidad se ordenan por nombre.

In [38]:
resultado_e = list(db_mongo["Citas"].aggregate([
    {"$match": {"paciente_rut": {"$ne": None}}},
    {
        "$lookup": {
            "from": "Estados",
            "localField": "estado_id",
            "foreignField": "_id",
            "as": "estado"
        }
    },
    {"$unwind": "$estado"},
    {"$match": {"estado.nombre": "Atendida"}},
    {
        "$lookup": {
            "from": "Especialidades",
            "localField": "especialidad_id",
            "foreignField": "_id",
            "as": "especialidad"
        }
    },
    {"$unwind": "$especialidad"},
    {
        "$group": {
            "_id": "$paciente_rut",
            "especialidades": {"$addToSet": "$especialidad.nombre"}
        }
    },
    {"$set": {"cantidad_especialidades": {"$size": "$especialidades"}}},
    {"$match": {"cantidad_especialidades": {"$gte": 3}}},
    {
        "$lookup": {
            "from": "Pacientes",
            "localField": "_id",
            "foreignField": "_id",
            "as": "paciente"
        }
    },
    {"$unwind": "$paciente"},
    {
        "$project": {
            "_id": 0,
            "paciente": "$paciente.nombre",
            "cantidad_especialidades": "$cantidad_especialidades",
            "especialidades": {
                "$sortArray": {"input": "$especialidades", "sortBy": 1}
            }
        }
    },
    {"$sort": {"cantidad_especialidades": -1, "paciente": 1}}
]))

mostrar_tabla(resultado_e)

sql_e = """
SELECT p.Nombre_Paciente AS paciente,
       COUNT(DISTINCT c.ID_Especialidad) AS cantidad_especialidades,
       GROUP_CONCAT(DISTINCT esp.Nombre_Especialidad
                    ORDER BY esp.Nombre_Especialidad SEPARATOR ', ') AS especialidades
FROM cita AS c
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
INNER JOIN paciente AS p ON p.RUT_Paciente = c.RUT_Paciente
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
WHERE e.Estado_Cita = 'Atendida'
GROUP BY p.RUT_Paciente, p.Nombre_Paciente
HAVING COUNT(DISTINCT c.ID_Especialidad) >= 3
ORDER BY cantidad_especialidades DESC, paciente
"""

print(sql_e)
comparar_con_mysql(resultado_e, sql_e)

| paciente | cantidad_especialidades | especialidades |
|---|---|---|
| Jorge Herrera | 4 | Cardiología, Gastroenterología, Medicina General, Neurología |


SELECT p.Nombre_Paciente AS paciente,
       COUNT(DISTINCT c.ID_Especialidad) AS cantidad_especialidades,
       GROUP_CONCAT(DISTINCT esp.Nombre_Especialidad
                    ORDER BY esp.Nombre_Especialidad SEPARATOR ', ') AS especialidades
FROM cita AS c
INNER JOIN estado AS e ON e.ID_Estado = c.ID_Estado
INNER JOIN paciente AS p ON p.RUT_Paciente = c.RUT_Paciente
INNER JOIN especialidad AS esp ON esp.ID_Especialidad = c.ID_Especialidad
WHERE e.Estado_Cita = 'Atendida'
GROUP BY p.RUT_Paciente, p.Nombre_Paciente
HAVING COUNT(DISTINCT c.ID_Especialidad) >= 3
ORDER BY cantidad_especialidades DESC, paciente

MySQL y MongoDB coinciden: 1 filas.


# 4.4. Replicación

Esta sección modifica los datos de MongoDB y detiene un nodo. Debe ejecutarse
una sola vez, en orden, sobre una base recién migrada.

## 4.4.1. Estado del replica set

Se consulta el estado desde Python y se identifica el rol de cada nodo.

In [39]:
def mostrar_roles():
    estado_rs = cliente_mongo.admin.command("replSetGetStatus")

    for miembro in estado_rs["members"]:
        print(f"{miembro['name']} → {miembro['stateStr']}")


print("Replica set:", cliente_mongo.admin.command("replSetGetStatus")["set"])
mostrar_roles()
print("Write concern de las escrituras:", db_mongo.write_concern.document)

Replica set: rsSaludUSM
mongo1:30001 → PRIMARY
mongo2:30002 → SECONDARY
mongo3:30003 → SECONDARY
Write concern de las escrituras: {'wtimeout': 15000, 'w': 'majority'}


## 4.4.2. Consistencia de los datos

Se crea una conexión directa a cada nodo para leer sus datos sin pasar por el
primario. Esas conexiones usan `directConnection=true`.

`write concern majority` confirma la escritura cuando la tiene la mayoría, es
decir, 2 de los 3 nodos. El tercero puede demorar unos milisegundos más, por lo
que la verificación reintenta durante un tiempo limitado hasta que los 3 nodos
retornen el mismo resultado. Si no lo logra, la celda falla.

Las reglas de negocio que en MySQL garantizan los triggers y las claves
foráneas aquí se validan en la aplicación. Cada operación actualiza en una
transacción tanto la colección principal como su copia en `Comprobantes` o
`Fichas`, para que ninguna quede desactualizada.

In [40]:
import time
from bson import json_util

NODOS = ["mongo1:30001", "mongo2:30002", "mongo3:30003"]

clientes_directos = {
    nodo: MongoClient(
        f"mongodb://{nodo}/?directConnection=true"
        "&readPreference=secondaryPreferred",
        serverSelectionTimeoutMS=5000,
        tz_aware=True
    )
    for nodo in NODOS
}

escritura_mayoritaria = WriteConcern(w="majority", wtimeout=15000)


def verificar_consistencia(coleccion, filtro, proyeccion=None, espera=20):
    """Consultar cada nodo de forma directa y comprobar que coincidan."""
    limite = time.monotonic() + espera

    while True:
        resultados = {
            nodo: list(
                cliente["salud_usm"][coleccion]
                .find(filtro, proyeccion)
                .sort("_id", 1)
            )
            for nodo, cliente in clientes_directos.items()
        }

        iguales = all(r == resultados[NODOS[0]] for r in resultados.values())

        if iguales or time.monotonic() > limite:
            break

        time.sleep(0.2)

    for nodo, documentos_nodo in resultados.items():
        rol = (
            clientes_directos[nodo].admin.command("hello").get("isWritablePrimary")
        )
        print(f"{nodo} ({'primario' if rol else 'secundario'}) | {coleccion}:")
        print("  " + json_util.dumps(documentos_nodo, ensure_ascii=False))

    assert resultados[NODOS[0]], "La consulta no retornó documentos."
    assert iguales, f"Los nodos no coinciden en {coleccion}."
    print(f"→ Los 3 nodos retornan el mismo resultado en {coleccion}.\n")

#### Operación 1: inserción de una nueva cita en estado Reservada

Antes de insertar se validan las reglas de la Tarea 2: el médico debe pertenecer
al centro y tener la especialidad, la fecha debe ser futura, y ni el médico ni el
paciente pueden tener otra cita vigente en ese bloque.

La transacción inserta la cita y su comprobante. Se usa una fecha fija de 2027
para que la cita siga siendo futura durante la defensa.

In [41]:
def reservar_cita(fecha, hora, rut_paciente, rut_medico, id_centro, id_especialidad):
    """Reservar una cita y su comprobante en una transacción."""

    def transaccion(sesion):
        medico = db_mongo["Medicos"].find_one(
            {
                "_id": rut_medico,
                "centro_ids": id_centro,
                "especialidad_ids": id_especialidad
            },
            session=sesion
        )
        if medico is None:
            raise ValueError("El médico no atiende esa especialidad en ese centro.")

        if fecha <= datetime.now(timezone.utc):
            raise ValueError("No se puede registrar una cita en una fecha pasada.")

        paciente = db_mongo["Pacientes"].find_one({"_id": rut_paciente}, session=sesion)
        if paciente is None:
            raise ValueError("El paciente no existe.")

        cancelada = db_mongo["Estados"].find_one({"nombre": "Cancelada"}, session=sesion)
        ocupado = db_mongo["Citas"].count_documents(
            {
                "fecha": fecha,
                "hora": hora,
                "estado_id": {"$ne": cancelada["_id"]},
                "$or": [
                    {"medico_rut": rut_medico},
                    {"paciente_rut": rut_paciente}
                ]
            },
            session=sesion
        )
        if ocupado:
            raise ValueError("El médico o el paciente ya tienen una cita en ese bloque.")

        reservada = db_mongo["Estados"].find_one({"nombre": "Reservada"}, session=sesion)
        prevision = db_mongo["Previsiones"].find_one(
            {"_id": paciente["prevision_id"]}, session=sesion
        )
        centro = db_mongo["Centros"].find_one({"_id": id_centro}, session=sesion)
        especialidad = db_mongo["Especialidades"].find_one(
            {"_id": id_especialidad}, session=sesion
        )
        ultima = db_mongo["Citas"].find_one(sort=[("_id", -1)], session=sesion)
        id_cita = ultima["_id"] + 1

        db_mongo["Citas"].insert_one(
            {
                "_id": id_cita,
                "fecha": fecha,
                "hora": hora,
                "estado_id": reservada["_id"],
                "paciente_rut": rut_paciente,
                "medico_rut": rut_medico,
                "especialidad_id": id_especialidad,
                "centro_id": id_centro
            },
            session=sesion
        )

        db_mongo["Comprobantes"].insert_one(
            {
                "_id": id_cita,
                "generado_en": datetime.now(timezone.utc),
                "cita": {
                    "id": id_cita,
                    "fecha": fecha,
                    "hora": hora,
                    "estado": {"id": reservada["_id"], "nombre": reservada["nombre"]},
                    "paciente": {
                        "rut": paciente["_id"],
                        "nombre": paciente["nombre"],
                        "telefono": paciente["telefono"],
                        "prevision": {"id": prevision["_id"], "tipo": prevision["tipo"]}
                    },
                    "medico": {
                        "rut": medico["_id"],
                        "nombre": medico["nombre"],
                        "especialidad": {
                            "id": especialidad["_id"],
                            "nombre": especialidad["nombre"]
                        }
                    },
                    "centro": {
                        "id": centro["_id"],
                        "nombre": centro["nombre"],
                        "comuna": centro["comuna"],
                        "region": centro["region"]
                    }
                }
            },
            session=sesion
        )

        return id_cita

    with cliente_mongo.start_session() as sesion:
        return sesion.with_transaction(
            transaccion,
            write_concern=escritura_mayoritaria
        )


fecha_nuevas_citas = datetime(2027, 1, 11, tzinfo=timezone.utc)

id_nueva_cita = reservar_cita(
    fecha_nuevas_citas, "10:00:00", "00000000-0", "00000000-1", 1, 4
)
print("Cita insertada:", id_nueva_cita, "\n")

verificar_consistencia("Citas", {"_id": id_nueva_cita})
verificar_consistencia(
    "Comprobantes",
    {"_id": id_nueva_cita},
    {"cita.fecha": 1, "cita.hora": 1, "cita.estado": 1}
)

Cita insertada: 206 

mongo1:30001 (primario) | Citas:
  [{"_id": 206, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "10:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
mongo2:30002 (secundario) | Citas:
  [{"_id": 206, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "10:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
mongo3:30003 (secundario) | Citas:
  [{"_id": 206, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "10:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
→ Los 3 nodos retornan el mismo resultado en Citas.

mongo1:30001 (primario) | Comprobantes:
  [{"_id": 206, "cita": {"fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "10:00:00", "estado": {"id": 0, "nombre": "Reservada"}}}]
mongo2:30002 (secundario) | Comprobantes:
  [{"_id": 206, "cita": {"

#### Operación 2: actualización de citas vencidas a Cancelada

Se seleccionan las citas Reservadas o Confirmadas con fecha anterior al día
actual en Chile, con el mismo criterio del procedimiento `CancelarCitasVencidas`
de MySQL: las citas de hoy siguen vigentes.

La cantidad de citas actualizadas depende de la fecha de ejecución.

In [42]:
def cancelar_citas_vencidas():
    """Cancelar las citas vencidas y actualizar sus comprobantes."""

    def transaccion(sesion):
        reservada = db_mongo["Estados"].find_one({"nombre": "Reservada"}, session=sesion)
        confirmada = db_mongo["Estados"].find_one({"nombre": "Confirmada"}, session=sesion)
        cancelada = db_mongo["Estados"].find_one({"nombre": "Cancelada"}, session=sesion)

        vencidas = db_mongo["Citas"].find(
            {
                "estado_id": {"$in": [reservada["_id"], confirmada["_id"]]},
                "$expr": {
                    "$lt": [
                        {
                            "$dateToString": {
                                "date": "$fecha",
                                "format": "%Y-%m-%d",
                                "timezone": "UTC"
                            }
                        },
                        {
                            "$dateToString": {
                                "date": "$$NOW",
                                "format": "%Y-%m-%d",
                                "timezone": "America/Santiago"
                            }
                        }
                    ]
                }
            },
            {"_id": 1},
            session=sesion
        )
        ids_vencidas = [documento["_id"] for documento in vencidas]

        if ids_vencidas:
            db_mongo["Citas"].update_many(
                {"_id": {"$in": ids_vencidas}},
                {"$set": {"estado_id": cancelada["_id"]}},
                session=sesion
            )
            db_mongo["Comprobantes"].update_many(
                {"_id": {"$in": ids_vencidas}},
                {
                    "$set": {
                        "cita.estado": {
                            "id": cancelada["_id"],
                            "nombre": cancelada["nombre"]
                        },
                        "generado_en": datetime.now(timezone.utc)
                    }
                },
                session=sesion
            )

        return ids_vencidas

    with cliente_mongo.start_session() as sesion:
        return sesion.with_transaction(
            transaccion,
            write_concern=escritura_mayoritaria
        )


ids_canceladas = cancelar_citas_vencidas()
print(f"Citas canceladas: {len(ids_canceladas)} → {ids_canceladas}\n")

verificar_consistencia(
    "Citas",
    {"_id": {"$in": ids_canceladas}},
    {"fecha": 1, "estado_id": 1}
)
verificar_consistencia(
    "Comprobantes",
    {"_id": {"$in": ids_canceladas}},
    {"cita.estado": 1}
)

Citas canceladas: 6 → [21, 28, 42, 44, 46, 47]



mongo1:30001 (primario) | Citas:
  [{"_id": 21, "fecha": {"$date": "2026-10-21T00:00:00Z"}, "estado_id": 4}, {"_id": 28, "fecha": {"$date": "2026-10-30T00:00:00Z"}, "estado_id": 4}, {"_id": 42, "fecha": {"$date": "2026-11-02T00:00:00Z"}, "estado_id": 4}, {"_id": 44, "fecha": {"$date": "2026-11-02T00:00:00Z"}, "estado_id": 4}, {"_id": 46, "fecha": {"$date": "2026-11-03T00:00:00Z"}, "estado_id": 4}, {"_id": 47, "fecha": {"$date": "2026-11-03T00:00:00Z"}, "estado_id": 4}]
mongo2:30002 (secundario) | Citas:
  [{"_id": 21, "fecha": {"$date": "2026-10-21T00:00:00Z"}, "estado_id": 4}, {"_id": 28, "fecha": {"$date": "2026-10-30T00:00:00Z"}, "estado_id": 4}, {"_id": 42, "fecha": {"$date": "2026-11-02T00:00:00Z"}, "estado_id": 4}, {"_id": 44, "fecha": {"$date": "2026-11-02T00:00:00Z"}, "estado_id": 4}, {"_id": 46, "fecha": {"$date": "2026-11-03T00:00:00Z"}, "estado_id": 4}, {"_id": 47, "fecha": {"$date": "2026-11-03T00:00:00Z"}, "estado_id": 4}]
mongo3:30003 (secundario) | Citas:
  [{"_id": 21, 

#### Operación 3: eliminación de una línea de receta

`$pull` elimina la línea del arreglo `recetas` de la atención y de la copia
guardada en su ficha.

Se elimina la receta 2 de la atención 1. Como la migración deja exactamente 80
recetas, el estado final contiene 79. Esto es consecuencia de la operación
solicitada; el mínimo de 80 se comprueba en la sección 4.2.

In [43]:
def eliminar_linea_receta(id_atencion, id_receta):
    """Eliminar una receta de la atención y de su ficha en una transacción."""

    def transaccion(sesion):
        resultado = db_mongo["Atenciones"].update_one(
            {"_id": id_atencion},
            {"$pull": {"recetas": {"receta_id": id_receta}}},
            session=sesion
        )
        if resultado.modified_count != 1:
            raise ValueError("La atención no contiene esa línea de receta.")

        db_mongo["Fichas"].update_one(
            {"_id": id_atencion},
            {
                "$pull": {"cita.atencion.recetas": {"receta_id": id_receta}},
                "$set": {"generado_en": datetime.now(timezone.utc)}
            },
            session=sesion
        )

    with cliente_mongo.start_session() as sesion:
        sesion.with_transaction(
            transaccion,
            write_concern=escritura_mayoritaria
        )


id_atencion_receta = 1
id_receta_eliminada = 2

eliminar_linea_receta(id_atencion_receta, id_receta_eliminada)
print(f"Receta {id_receta_eliminada} eliminada de la atención {id_atencion_receta}.\n")

verificar_consistencia(
    "Atenciones",
    {"_id": id_atencion_receta},
    {"recetas": 1}
)
verificar_consistencia(
    "Fichas",
    {"_id": id_atencion_receta},
    {"cita.atencion.recetas": 1}
)
print("Recetas totales:", contar_elementos("Atenciones", "recetas"))

Receta 2 eliminada de la atención 1.

mongo1:30001 (primario) | Atenciones:
  [{"_id": 1, "recetas": [{"receta_id": 1, "medicamento": "Paracetamol 500 mg", "dosis": "1 comprimido cada 8 horas", "dias": 5}, {"receta_id": 21, "medicamento": "Medicamento de prueba grupo 3", "dosis": "Pauta ficticia grupo 1", "dias": 3}]}]
mongo2:30002 (secundario) | Atenciones:
  [{"_id": 1, "recetas": [{"receta_id": 1, "medicamento": "Paracetamol 500 mg", "dosis": "1 comprimido cada 8 horas", "dias": 5}, {"receta_id": 21, "medicamento": "Medicamento de prueba grupo 3", "dosis": "Pauta ficticia grupo 1", "dias": 3}]}]
mongo3:30003 (secundario) | Atenciones:
  [{"_id": 1, "recetas": [{"receta_id": 1, "medicamento": "Paracetamol 500 mg", "dosis": "1 comprimido cada 8 horas", "dias": 5}, {"receta_id": 21, "medicamento": "Medicamento de prueba grupo 3", "dosis": "Pauta ficticia grupo 1", "dias": 3}]}]
→ Los 3 nodos retornan el mismo resultado en Atenciones.

mongo1:30001 (primario) | Fichas:
  [{"_id": 1, "ci

## 4.4.3. Alta disponibilidad

Se detiene el contenedor del nodo primario mediante Docker Compose. Luego se
reutilizan `cliente_mongo` y `db_mongo`, creados en la sección 4.2, sin modificar
su cadena de conexión.

El cliente espera automáticamente hasta que exista un primario y reintenta la
operación, por lo que el programa no debe manejar un error. Al detener el
contenedor, el primario cede su rol antes de apagarse. Por eso, la elección
puede terminar mientras se detiene y las operaciones siguientes no alcanzan a
esperar.

In [44]:
import subprocess

ARCHIVO_DEPLOY = "deploy-202473644-2-202473520-9.yml"


def docker_compose(accion, servicio):
    subprocess.run(
        ["docker", "compose", "-f", ARCHIVO_DEPLOY, accion, servicio],
        check=True,
        capture_output=True,
        text=True
    )


print("Roles antes de detener el primario:")
mostrar_roles()

primario_detenido = cliente_mongo.primary
servicio_detenido = primario_detenido[0]
nodo_detenido = f"{primario_detenido[0]}:{primario_detenido[1]}"

docker_compose("stop", servicio_detenido)
print(f"\nContenedor del primario detenido: {servicio_detenido}")

Roles antes de detener el primario:
mongo1:30001 → PRIMARY
mongo2:30002 → SECONDARY
mongo3:30003 → SECONDARY



Contenedor del primario detenido: mongo1


In [45]:
inicio = time.monotonic()
agenda_tras_falla = agenda_medico(rut_medico_agenda, fecha_agenda)
print(f"Consulta (a) completada en {time.monotonic() - inicio:.1f} s con el mismo cliente:")
mostrar_tabla(agenda_tras_falla)

inicio = time.monotonic()
id_cita_tras_falla = reservar_cita(
    fecha_nuevas_citas, "11:00:00", "00000000-0", "00000000-1", 1, 4
)
print(f"Cita {id_cita_tras_falla} insertada en {time.monotonic() - inicio:.1f} s con el mismo cliente.\n")

nuevo_primario = cliente_mongo.primary
print("Nuevo primario:", nuevo_primario)
assert nuevo_primario != primario_detenido

print("\nRoles con el nodo detenido:")
mostrar_roles()

print("\nAgenda con la cita insertada durante la falla:")
mostrar_tabla(agenda_medico(rut_medico_agenda, fecha_nuevas_citas))

Consulta (a) completada en 0.0 s con el mismo cliente:


| hora | paciente | prevision | centro | especialidad | estado |
|---|---|---|---|---|---|
| 09:00:00 | Martina Díaz | Isapre | SaludUSM San Joaquín | Medicina General | No Asistió |
| 10:30:00 | Jorge Herrera | Particular | SaludUSM Vitacura | Cardiología | Cancelada |
| 12:00:00 | Carlos Silva | Fonasa | SaludUSM San Joaquín | Medicina General | No Asistió |

Cita 207 insertada en 0.0 s con el mismo cliente.

Nuevo primario: ('mongo2', 30002)

Roles con el nodo detenido:
mongo1:30001 → SECONDARY
mongo2:30002 → PRIMARY
mongo3:30003 → SECONDARY

Agenda con la cita insertada durante la falla:


| hora | paciente | prevision | centro | especialidad | estado |
|---|---|---|---|---|---|
| 10:00:00 | Martina Díaz | Isapre | SaludUSM San Joaquín | Medicina General | Reservada |
| 11:00:00 | Martina Díaz | Isapre | SaludUSM San Joaquín | Medicina General | Reservada |

#### Reincorporación del nodo detenido

Se levanta nuevamente el contenedor y se consulta directamente ese nodo hasta
que vuelve como secundario y contiene la cita escrita durante su ausencia.
Como todos los nodos tienen la misma prioridad, no recupera el rol de primario.

In [46]:
from pymongo.errors import PyMongoError

docker_compose("start", servicio_detenido)
print(f"Contenedor iniciado: {servicio_detenido}")

cliente_nodo = clientes_directos[nodo_detenido]
limite = time.monotonic() + 120
reincorporado = False

while time.monotonic() < limite and not reincorporado:
    try:
        es_secundario = cliente_nodo.admin.command("hello").get("secondary", False)
        cita_replicada = cliente_nodo["salud_usm"]["Citas"].find_one(
            {"_id": id_cita_tras_falla}
        )
        reincorporado = es_secundario and cita_replicada is not None
    except PyMongoError:
        pass

    if not reincorporado:
        time.sleep(1)

assert reincorporado, "El nodo no se reincorporó con los datos esperados."

print(f"{nodo_detenido} se reincorporó como secundario.\n")
mostrar_roles()
print()

verificar_consistencia("Citas", {"_id": id_cita_tras_falla})
verificar_consistencia(
    "Comprobantes",
    {"_id": id_cita_tras_falla},
    {"cita.fecha": 1, "cita.hora": 1, "cita.estado": 1}
)

Contenedor iniciado: mongo1


mongo1:30001 se reincorporó como secundario.

mongo1:30001 → SECONDARY
mongo2:30002 → PRIMARY
mongo3:30003 → SECONDARY

mongo1:30001 (secundario) | Citas:
  [{"_id": 207, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "11:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
mongo2:30002 (primario) | Citas:
  [{"_id": 207, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "11:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
mongo3:30003 (secundario) | Citas:
  [{"_id": 207, "fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "11:00:00", "estado_id": 0, "paciente_rut": "00000000-0", "medico_rut": "00000000-1", "especialidad_id": 4, "centro_id": 1}]
→ Los 3 nodos retornan el mismo resultado en Citas.

mongo1:30001 (secundario) | Comprobantes:
  [{"_id": 207, "cita": {"fecha": {"$date": "2027-01-11T00:00:00Z"}, "hora": "11:00:00", "estado": {"id

#### Estado final de las colecciones

Se comprueba que las copias sigan coincidiendo con las colecciones principales
después de todas las operaciones.

In [47]:
for coleccion in [
    "Centros", "Especialidades", "Previsiones", "Estados", "Diagnosticos",
    "Pacientes", "Medicos", "Citas", "Atenciones", "Comprobantes", "Fichas"
]:
    print(f"{coleccion}: {db_mongo[coleccion].count_documents({})}")

assert (
    db_mongo["Citas"].count_documents({})
    == db_mongo["Comprobantes"].count_documents({})
)
assert (
    db_mongo["Atenciones"].count_documents({})
    == db_mongo["Fichas"].count_documents({})
)

for cliente in clientes_directos.values():
    cliente.close()

Centros: 5
Especialidades: 8
Previsiones: 3
Estados: 5
Diagnosticos: 14
Pacientes: 49
Medicos: 22
Citas: 207
Atenciones: 60
Comprobantes: 207
Fichas: 60
